# GC-Mamba Tensor Explorer

This notebook loads two frozen-from-vanilla GC-Mamba checkpoints, extracts the learned temporal/Mamba tensors, and gives quick tools for printing tensors and plotting singular-value, also called Schmidt-value, spectra.

Run this notebook from the GraphCast environment used by the repo.

In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "scripts/graphcast_env.sh").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find Weather_global repo root from current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.graphcast.training.core.model import load_graphcast_checkpoint

BASE_DIR = ROOT / "artifacts/checkpoints/7_years/mamba_frozen_from_vanilla_mp6_20k_res2_target_steps_bptt16"
CHECKPOINT_FILE = "ckpt_best.npz"

RUNS = {
    "di64_ds32": BASE_DIR / "vanilla_gc_7y_res2_m4_w512_mp6_h6_bs8_accum1_stream50k_gc_mamba_tc2_di64_ds32_20k_target_step4_bptt16",
    "di128_ds64": BASE_DIR / "vanilla_gc_7y_res2_m4_w512_mp6_h6_bs8_accum1_stream50k_gc_mamba_tc2_di128_ds64_20k_target_step4_bptt16",
}

CKPT_PATHS = {label: run_dir / CHECKPOINT_FILE for label, run_dir in RUNS.items()}
CKPT_PATHS

## What Is In These Tensors?

These runs are GC-Mamba models initialized from a vanilla GraphCast checkpoint with `trainable_part = mamba`. The vanilla GraphCast parameters were frozen; the temporal/Mamba parameters were learned.

The temporal block is inserted at two interleaved mesh-processor locations, visible in tensor paths as `mesh_interleaved_temporal_r0_s0` and `mesh_interleaved_temporal_r0_s1`. Each insertion has one `mamba_block_0` plus its local layer norm.

Core matrix-like tensors include `in_proj/w`, `out_proj/w`, `~ssm/x_proj/w`, `~ssm/dt_proj/w`, `A_log`, and `conv1d/kernel`. One-dimensional tensors like `D`, biases, layernorm scale, and layernorm offset are useful to inspect, but are skipped by SVD plots unless reshaped explicitly.

In [ ]:
INSERTION_RE = re.compile(r"mesh_interleaved_temporal_r(?P<rep>\d+)_s(?P<step>\d+)")


def load_run_config(run_dir: Path) -> dict:
    with (run_dir / "run_config.json").open("r", encoding="utf-8") as f:
        return json.load(f)


def tensor_kind(path: str) -> str:
    if "layer_norm" in path:
        return "layernorm"
    if "mamba_block" in path:
        return "mamba"
    return "temporal"


def insertion_name(path: str) -> str:
    match = INSERTION_RE.search(path)
    if not match:
        return "unknown"
    return f"r{match.group('rep')}_s{match.group('step')}"


def short_tensor_name(path: str) -> str:
    marker = "~_run_sequence/"
    if marker in path:
        return path.split(marker, 1)[1]
    return path


def iter_param_leaves(params):
    for module_name, module_params in params.items():
        for param_name, value in module_params.items():
            yield module_name, param_name, value


def extract_mamba_tensors(ckpt, include_layernorm: bool = True) -> dict[str, np.ndarray]:
    tensors: dict[str, np.ndarray] = {}
    for module_name, param_name, value in iter_param_leaves(ckpt.params):
        full_path = f"{module_name}/{param_name}"
        lower = full_path.lower()
        is_temporal = "temporal" in lower or "mamba" in lower
        is_ln = "layer_norm" in lower
        if not is_temporal:
            continue
        if is_ln and not include_layernorm:
            continue
        tensors[full_path] = np.asarray(value)
    return dict(sorted(tensors.items()))


def finite_stats(arr: np.ndarray) -> dict:
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    if not finite.all():
        values = arr[finite]
    else:
        values = arr
    if values.size == 0:
        return {"finite": False, "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan, "fro_norm": np.nan}
    return {
        "finite": bool(finite.all()),
        "min": float(values.min()),
        "max": float(values.max()),
        "mean": float(values.mean()),
        "std": float(values.std()),
        "fro_norm": float(np.linalg.norm(values.reshape(-1))),
    }


def summarize_tensors(all_tensors: dict[str, dict[str, np.ndarray]]) -> pd.DataFrame:
    rows = []
    for run_label, tensors in all_tensors.items():
        for path, arr in tensors.items():
            stats = finite_stats(arr)
            rows.append(
                {
                    "run": run_label,
                    "insertion": insertion_name(path),
                    "kind": tensor_kind(path),
                    "tensor": short_tensor_name(path),
                    "path": path,
                    "shape": tuple(arr.shape),
                    "dtype": str(arr.dtype),
                    "params": int(arr.size),
                    **stats,
                }
            )
    return pd.DataFrame(rows)


def print_tensor(run_label: str, tensor_pattern: str, max_items: int = 20) -> np.ndarray:
    matches = [key for key in ALL_TENSORS[run_label] if tensor_pattern in key]
    if not matches:
        raise KeyError(f"No tensor in {run_label!r} matched {tensor_pattern!r}")
    if len(matches) > 1:
        print("Multiple matches. Printing the first one; refine the pattern if needed:")
        for key in matches:
            print("  ", key)
    key = matches[0]
    arr = ALL_TENSORS[run_label][key]
    flat = arr.reshape(-1)
    print(f"{run_label}: {key}")
    print(f"shape={arr.shape}, dtype={arr.dtype}, params={arr.size}")
    print(flat[:max_items])
    return arr

In [ ]:
RUN_CONFIGS = {label: load_run_config(run_dir) for label, run_dir in RUNS.items()}
CHECKPOINTS = {label: load_graphcast_checkpoint(path) for label, path in CKPT_PATHS.items()}
ALL_TENSORS = {label: extract_mamba_tensors(ckpt, include_layernorm=True) for label, ckpt in CHECKPOINTS.items()}

for label, tensors in ALL_TENSORS.items():
    temporal_cfg = RUN_CONFIGS[label]["temporal_config"]
    print(label, CKPT_PATHS[label])
    print("  temporal_config:", {k: temporal_cfg[k] for k in ["d_inner", "d_state", "d_conv", "layers", "insert_count", "stateful"]})
    print("  extracted leaves:", len(tensors))

In [ ]:
SUMMARY = summarize_tensors(ALL_TENSORS)
SUMMARY.sort_values(["run", "insertion", "kind", "tensor"])

In [ ]:
# Tensor paths available for filtering/printing.
for run_label, tensors in ALL_TENSORS.items():
    print("\n" + run_label)
    for key in tensors:
        print("  ", key)

In [ ]:
# Examples: change the string pattern to inspect any tensor above.
_ = print_tensor("di64_ds32", "r0_s0/~_run_sequence/mamba_block_0/in_proj/w", max_items=24)
_ = print_tensor("di128_ds64", "r0_s0/~_run_sequence/mamba_block_0/A_log", max_items=24)

In [ ]:
def matrix_view(tensor: np.ndarray, mode: str = "auto") -> np.ndarray | None:
    arr = np.asarray(tensor, dtype=np.float64)
    if mode == "auto":
        if arr.ndim < 2:
            return None
        if arr.ndim == 2:
            return arr
        return arr.reshape(arr.shape[0], -1)
    if mode == "first_dim":
        return arr.reshape(arr.shape[0], -1)
    if mode == "last_dim":
        return arr.reshape(-1, arr.shape[-1])
    if mode == "vector_column":
        return arr.reshape(-1, 1)
    raise ValueError(f"Unknown matrix_view mode: {mode}")


def svd_values(tensor: np.ndarray, mode: str = "auto", normalize: bool = False) -> np.ndarray:
    mat = matrix_view(tensor, mode=mode)
    if mat is None:
        return np.array([], dtype=np.float64)
    values = np.linalg.svd(mat, compute_uv=False)
    if normalize and values.size and values[0] != 0:
        values = values / values[0]
    return values


def selected_tensors(
    all_tensors: dict[str, dict[str, np.ndarray]],
    pattern: str | None = None,
    *,
    include_layernorm: bool = False,
) -> dict[tuple[str, str], np.ndarray]:
    selected = {}
    for run_label, tensors in all_tensors.items():
        for path, arr in tensors.items():
            if pattern is not None and pattern not in path:
                continue
            if not include_layernorm and tensor_kind(path) == "layernorm":
                continue
            if matrix_view(arr) is None:
                continue
            selected[(run_label, path)] = arr
    return selected


def plot_svd(
    all_tensors: dict[str, dict[str, np.ndarray]],
    pattern: str | None = None,
    *,
    normalize: bool = False,
    logy: bool = True,
    mode: str = "auto",
    max_curves: int | None = None,
) -> None:
    chosen = selected_tensors(all_tensors, pattern=pattern)
    if max_curves is not None:
        chosen = dict(list(chosen.items())[:max_curves])
    if not chosen:
        raise ValueError("No matrix-like tensors matched the selection.")

    plt.figure(figsize=(10, 6))
    for (run_label, path), arr in chosen.items():
        values = svd_values(arr, mode=mode, normalize=normalize)
        if values.size == 0:
            continue
        label = f"{run_label} | {insertion_name(path)} | {short_tensor_name(path)}"
        plt.plot(np.arange(1, values.size + 1), values, marker=".", linewidth=1.2, label=label)

    plt.xlabel("Singular value index")
    plt.ylabel("Normalized Schmidt value" if normalize else "Schmidt value")
    if logy:
        plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=1)
    plt.tight_layout()
    plt.show()


def plot_schmidt_distribution(
    all_tensors: dict[str, dict[str, np.ndarray]],
    pattern: str | None = None,
    *,
    normalize: bool = True,
    bins: int = 50,
    mode: str = "auto",
) -> pd.DataFrame:
    rows = []
    for (run_label, path), arr in selected_tensors(all_tensors, pattern=pattern).items():
        values = svd_values(arr, mode=mode, normalize=normalize)
        for value in values:
            rows.append(
                {
                    "run": run_label,
                    "insertion": insertion_name(path),
                    "tensor": short_tensor_name(path),
                    "schmidt_value": float(value),
                }
            )
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No singular values matched the selection.")

    plt.figure(figsize=(10, 5))
    for run_label, group in df.groupby("run"):
        plt.hist(group["schmidt_value"], bins=bins, alpha=0.45, density=True, label=run_label)
    plt.xlabel("Normalized Schmidt value" if normalize else "Schmidt value")
    plt.ylabel("Density")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    return df

In [ ]:
# Compare individual tensor families across both runs and both insertions.
plot_svd(ALL_TENSORS, pattern="in_proj/w", normalize=True, logy=True)
plot_svd(ALL_TENSORS, pattern="out_proj/w", normalize=True, logy=True)
plot_svd(ALL_TENSORS, pattern="~ssm/x_proj/w", normalize=True, logy=True)
plot_svd(ALL_TENSORS, pattern="~ssm/dt_proj/w", normalize=True, logy=True)

In [ ]:
# All matrix-like Mamba tensors together. Set pattern="A_log" or pattern="conv1d/kernel" to narrow it.
schmidt_df = plot_schmidt_distribution(ALL_TENSORS, pattern=None, normalize=True, bins=60)
schmidt_df.head()